## Pandas Cheatsheet 
- Pandas is a fast, powerful and flexible open-source Python library for data manipulation and analysis 
    - Load clean and transform datasets
    - Filter, group, and aggregate
    - Merge, reshape and export datasets
- Provides two core data structures 
    - Series: one dimensional labeled array 
    - DataFrame: 2-dimensional labeled table (e.g. spreadsheet or SQL table)
- We can define a function e.g. process(x): return x+1, then df.apply(process, axis=0). Very fas vs using for loops in Python (e.g. for i in range(1,10): ...)
- However, in Big data situations, may not be able to use regular python pandas library as data structure is memory intensive. Hence can use Spark 
- Further, Pandas is designed for 2D tabular data (e.g. like excel or SQL table of Python). Hence, it is best for 
    - Data cleaning and preprocessing 
    - Exploratory Data Analysis (EDA)
    - Tabular datasets (CSV, Excel)
- In deep learning (like in PyTorch), we work on Tensors (multi-dim numerical arrays) not tables during training time and testing time
    - PyTorch has its own data structure torch.Tensor
- In some cases, can use both together 
    - df = pd.read_csv("data.csv")
    - X = torch.tensor(df[["feature1", "feature2"]].values, dtype=torch.float32)
    - y = torch.tensor(df['target'].values, dtype=torch.long) # recall that torch.long is PyTorch's way of specifying integers, used when feeding indices or targets into loss functions e.g. CrossEntropyLoss
        - e.g. loss_fn = torch.nn.CrossEntropyLoss()

In [ ]:
import pandas as pd

# Create a Pandas DataFrame from a dictionary
data = {'Name': ['John', 'Mary', 'Sarah'], 
        'Age': [25, 32, 28]}
df = pd.DataFrame(data) 
print(df)

In [ ]:
#Reading and writing files of various formats 

## CSV
df = pd.read_csv("data.csv")
df.to_csv("output.csv", index=False)

## Excel 
df = pd.read_excel("data.xlsx")
df.to_excel("output.xlsx")

##Json 
df = pd.read_json("data.json")
df.to_json("output.json")

In [ ]:
#Exploring data 
df.head() #displays first 5 rows 
df.tail() #displays last 5 rows 

df.shape #(rows, columns). This is an attribute, not a method. 
df.info() #summary of columns, dtypes, nulls
df.describe() #summary stats for numeric columns 
df.dtypes #data types of each column

df.columns #returns Pandas index object, not plain list, of column names 
list(df.columns)
df.index #returns row labels of dataframe as a Pandas Index Object, not a regular Python list.
list(df.index) #returns list

In [ ]:
#Selecting and Filtering . Follows same convention in Python of using [] for indexing/slicing operations 
## Further note: Both Slicing and Filtering selects subsets of a database, but both are conceptually different. 
### Slicing: selecting rows and columns using positions or ranges (df[0:5])
### Filtering: selecting rows based on a condition (df[df[age]>30])

df['col'] #select column as series 
df[['col1', 'col2']] #select multiple columns 

df.loc[0] #select row by label/index
df.loc[0, 'col'] #row 0, column 'col']

df.iloc[0] #select row by position 
df.iloc[0,1] #Row 0, column 1

df[df['age'] > 30] #filtering rows based on a condition applied to a column's values. df['age']>30 returns a Boolean Series, then df[] uses this Boolean series to filter rows, only keeping the ones where condition is True 
df[(df['age'] > 30) & (df['name'] == 'Charlie')] #filters rows where age >30 and name=='Charlie'. Use ampersand "&"" for bitwise logic applied to a Boolean series, which compares both series element-by-element and returns True only when both conditions are True for the same row
df[(df['age'] > 30) | (df['name'] == 'Charlie')] #same as the above, but use | for or 
df[~((df['age'] > 30) | (df['name'] == 'Charlie'))] #wraps the entire expression in parenthesis and prepends ~ to indicate not

#remember to wrap in () so that Python knows precedence of operations
#and/or are scalar logical operators e.g. x = True, y = False, (x and y), (x or y)
#if use these, Python will say that the truth value of a Series is ambiguous 

df.query('age>30') #query using a string


In [ ]:
#Modifying Data 
import numpy as np

#Creating new columns 
df['new_col'] = df['age'] * 2  #can just create new column by direct assignment. If column alr exists, it gets overwritten. Each value in this new column is the result of multiplication of original column by 2 element-wise
df['age_group'] = df['age'].apply(lambda x: 'young' if x < 30 else 'old') #using lambda expressions with .apply() that goes thru the df['age'] series element by element, x is dummy variable representing the current element in the series, and then the functions assigns strings 'young' or 'old' depending on whether the condition is met
df['is_senior'] = df['age'] > 30 #using conditionals, returns a Boolean series where each row checks if the age is greater than 30


df.rename(columns={'old':'new'}, inplace=True) #the columns={} renames specific columns by name, and inplace=True applies changes to the original df

df.drop('col', axis=1, inplace=True) #drop a column, which is axis=1
df.drop(0, axis=0, inplace=True) #drop a row by index

df.fillna(value=0, inplace=True) #fill missing values with 0. can specify other values to fill the Nans
df.fillna({'age': 0, 'score': 99})  # Fill different values per column by specifying dictionary where keys = column name, value = desired value

df.dropna(axis=1, inplace=True) #drop missing values along the columns, with threshold set as having at least 2 non-NAN values
df.replace('unknown', np.nan, inplace=True) #Replace string 'unknown' with Nan
df.replace({0: np.nan, -1: None}, inplace=True) #replace multiple values by passing in dict

#note: if i don't use inplace=True, then Python will return a new modified copy of the DataFrame without changing the original
#if we don't assign anything, then the modified version is returned by the function but immediately discarded unless I assign it
#hence the two ways to keep the result are (1) use inplace=True, or (2) assign to a new variable

#also: on axis: axis=0 operates along rows (e.g. scans, sums), whereas axis=1 operates along columns

In [ ]:
#Other useful data modifying code/functions 

## Modifying column datatypes
df['age'] = df['age'].astype(int) #convert to int
df['joined'] = pd.to_datetime(df['joined'])

## Check for missing values 
df.isna().sum() #count missing values per column 
df[df['age'].isna()] #filter rows where age is missing
df[df['score'].notna()] #keep rows where score is not missing

## Finding and dropping duplicated rows to help clean datasets with repeated records
df[df.duplicated()] #find duplicate rows
df.drop_duplicates(inplace=True) #drop exact duplicate rows
df.drop_duplicates(subset='id') #drop based on a specific column

## Limiting values to a min/max range
df['score'] = df['score'].clip(lower=0, upper=100) #caps out-of-bounds values, value<lower = lower, value>upper = upper, lower < value < upper = value

In [ ]:
#Combining DataFrames 
pd.concat([df1, df2], axis=0, ignore_index=False) #stack vertically (i.e. by rows) by passing in a list of dfs. if axis=1: combine columns
pd.merge(df1, df2, on='key', how='inner', indicator=True) #sql style join. on specifices the common column to join on, how refers to the type of join (inner, outer, left, right), indicator=True adds a column showing origin of each row

## More on types of joins
### inner: keeps only matching rows in both 
### outer: keeps all rows, fills missing with NaN 
### left: keeps all rows from df1, only matches from df2
### right: keeps all rows from df2, only matches from df1

In [ ]:
#Grouping and Aggregating 
df.groupby('col').mean() #conceptually, this (1) splits the dataframe into groups based on the unique values of a column, applies aggregation functions (e.g. mean, sum, count), and combines the results into a new DataFrame
df.groupby('col').agg(['mean', 'sum']) #multiple metrics per column
#note: df.groupby('col').mean(), the .mean() is operating on all numeric columns in the DataFrame except the one you grouped by
# for this df.groupby('col').agg(['mean', 'sum']), aggregation functions applied to all numeric columns 


df.groupby('col').agg({ #splits the DF into groups based on unique values in col (e.g. categories), then .agg applies diff agg functions to diff columns using dict
    'score': ['mean', 'max', 'std'], #for each group, score will be aggregated based on mean, max, std
    'age': 'median' #age will be aggregated by median
})
# score and age are just other columns in the DF, and if i only include some columns in .agg(), only those columns are aggregated in the output whereas the rest are ignored


##More notes
### general syntax: df.groupby('col').agg(func)
### Other aggregation functions: mean, sum, count, min, max, median, std, nunique

In [ ]:
#Sorting and rearranging 
df.sort_values(by = 'age', ascending=True, inplace=False, na_positions='last') #sorts the dataframe by the age column in ascending order (default). Original is unchanged unless assigned
df.sort_values(['age', 'name']) #sort by multiple
df.reset_index(drop=True) #reset index to default (0,1,2,...). Useful after filtering, grouping, or sorting
df_filtered = df[df['score'] > 80].reset_index(drop=True) #i.e. after filtering, the row index is no longer in running order, it will be missing values that were filtered out. Hence we can reset index. drop=True means we don't keep the old index as a column, just throw away. drop=False means we keep the old index as a new column in the DataFrame
df.set_index('name', inplace=False, drop=False) #set a new index by specifying the column name

In [ ]:
#Apply Functions 
df['col'].apply(lambda x: x*2) #element-wise operation 
df.applymap(str.upper) #apply to the entire DataFrame
df['col'].map({'A':1, 'B':2})

In [ ]:
#Iterating through rows in dataframe. Method .iterrows(): returns each row as a Pandas Series object, where "index" is the row index, and "row" is the actual data in the row
for index, row in df.iterrows(): #index will return the actual index of the dataframe (default is 0,1,2 unless otherwise specified)
    print(row['Name'], row['Age']) #index of each row series is the column names, and the values of the series is the cell values in that row

#df.apply() applies a function along either axis (rows or columns) of a DataFrame. Much more efficient than iterrows() as it leverages vectorized oeprations internally. Can also weave in custom functions 
#syntax: df.apply(func, axis=0)

data = {'A': [1, 2, 3], #store data as a dictionary where the column names are the keys, and the values are the data for each column
        'B': [4, 5, 6]}

df = pd.DataFrame(data, index=['a', 'b', 'c'])

df_sum = df.apply(sum)  # Adds values in each column
print(df_sum)


In [ ]:
# CSV File - Write and read back the DataFrame. 
df = pd.read_csv(
    'data.csv',               # filepath or buffer
    sep=',',                  # Column separator (default: ',')
    header=0,                 # Row number to use as column names (default: 0)
    names=['Col1', 'Col2'],   # Override column names
    index_col=0,              # Column to use as the row labels (index)
    usecols=['Col1', 'Col2'], # Subset of columns to read
    dtype={'Col1': int, 'Col2': float},  # Specify data types
    engine='python',          # Parser engine to use (Python or C)
    skiprows=2,               # Skip the first 2 rows
    skipfooter=1,             # Skip the last row (needs engine='python')
    nrows=100,                # Read only the first 100 rows
    na_values=['NA', 'null'], # Recognize these as NaN
    keep_default_na=True,     # Keep default NaN values along with user-defined
    parse_dates=['Col1'],     # Attempt to parse Col1 as date
    dayfirst=True,            # Treat the first value as the day when parsing dates
    infer_datetime_format=True, # Try to infer the date format
    encoding='utf-8',         # File encoding
    compression='infer',      # Automatically detect compression (gzip, zip, etc.)
    thousands=',',            # Handle commas as thousands separators
    decimal='.',              # Character for decimal points
    error_bad_lines=False,    # Skip lines with too many fields (deprecated)
    warn_bad_lines=True,      # Show warnings for lines with errors (deprecated)
    skip_blank_lines=True,    # Skip blank lines
    comment='#',              # Ignore lines starting with #
    delim_whitespace=False,   # Use whitespace as delimiter (True or False)
    low_memory=True,          # Process the file in chunks (useful for large files)
    memory_map=False,         # Map file to memory to improve performance
    float_precision='high',   # Handle floating point precision
    mangle_dupe_cols=True     # Duplicate columns will be renamed
)

In [ ]:
#writing dataframes to csv 
df.to_csv(
    'output.csv',          # Path or buffer to write to
    sep=',',               # Field delimiter (default: ',')
    index=True,            # Write row names (default is True)
    header=True,           # Write column names (default is True)
    columns=['A', 'B'],    # Write only a subset of columns
    mode='w',              # File mode (default is 'w')
    encoding='utf-8',      # File encoding (default: 'utf-8')
    quoting=1,             # Control how quotes are handled (QUOTE_MINIMAL)
    quotechar='"',         # Character used to quote fields
    line_terminator='\n',  # Character(s) used to end lines
    decimal='.',           # Character recognized as decimal point
    float_format='%.2f',   # Format string for floating-point numbers
    date_format='%Y-%m-%d',# Format string for datetime objects
    compression='infer',   # Compression mode (e.g., 'gzip', 'bz2', 'zip', 'xz')
    chunksize=1000,        # Write file in chunks of rows
    mode='a',              # Append to the file instead of overwriting
    errors='strict',       # Error handling for bad encoding (strict, replace, etc.)
    storage_options=None   # Extra options for remote file systems
)

In [ ]:
### Key plotting functions built into pandas -> df.plot()
df.plot(x='Col1', #Column for x-axis, i.e. the column names in the dataframe. If omit the x parameter, Pandas will automatically use the DF index as the x-axis
        y=['Col2', 'Col3'], #Column for y-axis, ditto above but can pass in a list of columns for multiple lines
        kind='line', #Type of plot (others include bar, barh, hist, box, kde, area, pie, scatter, hexbin)
        title='Line Plot', 
        xlabel='xlabel_name', #name for the x-axis label 
        ylabel='ylabel_name', #desired name for the y-axis label 
        figsize=(10,6), 
        grid=False, #if True, then shows background gridlines to help align eye visually when comparing values
        legend=False #default True when plotting multiple lines, and will automatically be Col2 and Col3 names 
        )


### 🐼 Summary of key attributes and methods 

A quick reference for working with Pandas DataFrames — covering key **attributes** and **methods**, including common arguments and their default values.

---

#### 📦 ATTRIBUTES (No `()` — direct access)

| Attribute     | Description                                | Example             |
|--------------|--------------------------------------------|---------------------|
| `df.shape`    | Tuple of (rows, columns)                    | `(100, 5)`          |
| `df.columns`  | Index of column names                       | `Index(['A', 'B'])` |
| `df.index`    | Index of row labels                         | `RangeIndex(...)`   |
| `df.dtypes`   | Data types of each column                   | `A: int64`          |
| `df.values`   | Data as a NumPy array                       | `array([[...],...])`|
| `df.size`     | Total elements = rows × columns             | `500`               |
| `df.ndim`     | Number of dimensions (usually 2)            | `2`                 |

---

#### ⚙️ METHODS (Use with `()` — performs action)

| Method | Description | Common Arguments (with Defaults) | Example |
|--------|-------------|----------------------------------|---------|
| `df.head()` | Return first `n` rows | `n=5` | `df.head()` |
| `df.tail()` | Return last `n` rows | `n=5` | `df.tail(3)` |
| `df.info()` | Summary of DataFrame: index, columns, dtypes, nulls | — | `df.info()` |
| `df.describe()` | Summary statistics for numeric columns | `percentiles=[.25, .5, .75]`, `include=None`, `exclude=None` | `df.describe(include='all')` |
| `df.isnull()` | Boolean DataFrame: True where values are NaN | — | `df.isnull()` |
| `df.notnull()` | Opposite of `isnull()` | — | `df.notnull()` |
| `df.dropna()` | Drop rows or columns with NaNs | `axis=0`, `how='any'`, `thresh=None`, `subset=None`, `inplace=False` | `df.dropna(subset=['age'])` |
| `df.fillna()` | Fill missing values | `value=None`, `method=None`, `axis=None`, `inplace=False`, `limit=None` | `df.fillna(0)` |
| `df.sort_values()` | Sort by column(s) | `by`, `axis=0`, `ascending=True`, `inplace=False`, `na_position='last'` | `df.sort_values(by='score')` |
| `df.sort_index()` | Sort by index | `axis=0`, `ascending=True`, `inplace=False` | `df.sort_index()` |
| `df.rename()` | Rename columns or index labels | `columns=None`, `index=None`, `inplace=False` | `df.rename(columns={'old':'new'})` |
| `df.groupby()` | Group by column(s) | `by`, `axis=0`, `as_index=True`, `sort=True` | `df.groupby('region')` |
| `df.apply()` | Apply function along axis | `func`, `axis=0` (`0`=columns, `1`=rows) | `df.apply(np.sqrt)` |
| `df.astype()` | Cast column(s) to a new type | `dtype`, `copy=True`, `errors='raise'` | `df.astype({'col':'float'})` |

---

🧠 **Tips**
- `axis=0` → operate **down columns** (i.e. for each column, do this)
- `axis=1` → operate **across rows** (i.e. for each row, do this)
- `inplace=True` → modifies the DataFrame directly (no new object returned)

---

